In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
%%writefile /kaggle/working/my_agent.py
"""
Go-Explore agent for ARC-AGI-3.

Key insight (from experiments on ls20/re86/g50t):
  Pure curiosity (RND/ICM) NEVER sees L2/L3 rewards — the reward requires
  a specific action sequence too long to stumble upon randomly. This is a
  credit-assignment problem, not a saturation problem.

Go-Explore solution — BFS over the state-action space:
  Each 'life' (actions between RESETs, ~50-130/game depending on energy budget):
    1. REPLAY path to the BFS-frontier (shallowest archived state with untried actions)
    2. TRY one untried action from the frontier
    3. ARCHIVE the new state with its path from level start
  Guaranteed to find any reachable goal state within SAFE_DEPTH steps.

Verified locally:
  * RESET -> current level start (not game start)     <- perfect for Go-Explore
  * Replay is fully DETERMINISTIC                     <- same actions = same state
  * ls20 L1 cleared in 9,137 steps / 69 episodes     <- 2x better than leaky RND
  * Energy/episode budget: ls20~129, tu93~50, re86~100, g50t~130
"""
import random
from collections import deque
from typing import Any

import numpy as np
from arcengine import FrameData, GameAction, GameState
from agents.agent import Agent

ACTION_MAP = {i: getattr(GameAction, f'ACTION{i}') for i in range(1, 8)}
ACTION_MAP[0] = GameAction.RESET

# Conservative safe depth per game (leave room for exploration after replay).
# Verified: ls20 with safe_depth=100 clears L1 at 9137 steps (ep=69).
# Rule of thumb: ~0.75 * episode_budget.
SAFE_DEPTH = {'ls20': 100, 'tu93': 40, 're86': 80, 'g50t': 100, 'default': 70}


def board_hash(fd: FrameData) -> int:
    """Stable hash of board state (timer rows 60-63 masked out)."""
    b = np.array(fd.frame, dtype=np.int8)[-1].copy()
    b[60:] = 0
    return hash(b.tobytes())


class GoExploreAgent(Agent):
    """
    BFS Go-Explore for ARC-AGI-3.

    Archive: state_hash -> {path: [action_ints from level start],
                             untried: set(action_ints not yet tried from here)}
    bfs_q:   deque of hashes in discovery order (shallowest states first)

    Per episode:
      1. pick_frontier() -> shallowest state with untried actions (within safe_depth)
      2. RESET -> replay the stored path to reach it
      3. try untried action -> archive new state
      4. keep exploring until GAME_OVER
      5. repeat
    """

    MAX_ACTIONS = float('inf')

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)

        self.archive: dict[int, dict] = {}
        self.bfs_q:   deque[int]      = deque()

        self.curr_path:  list[int] = []
        self.replay_seq: list[int] = []
        self.replay_idx: int       = 0
        self.in_replay:  bool      = False

        game_key = self.game_id.split('-')[0]
        self.safe_depth: int = SAFE_DEPTH.get(game_key, SAFE_DEPTH['default'])

        self.current_level: int = 0
        self.episode:       int = 0
        self.total_steps:   int = 0

        print(f'[GoExplore] game={self.game_id} safe_depth={self.safe_depth}', flush=True)

    # ── required interface ────────────────────────────────────────────────────

    def is_done(self, frames, latest_frame):
        # Play until all levels complete (WIN). Level clears detected in choose_action.
        return latest_frame.state is GameState.WIN

    def choose_action(self, frames, latest_frame):
        # ── level advance: clear archive, start fresh BFS for new level ───────
        lvl = int(getattr(latest_frame, 'levels_completed', 0) or 0)
        if lvl > self.current_level:
            print(f'[GoExplore] level {self.current_level}→{lvl} '
                  f'ep={self.episode} arch={len(self.archive)} '
                  f'steps={self.total_steps}', flush=True)
            self.archive.clear(); self.bfs_q.clear()
            self.curr_path = []; self.replay_seq = []
            self.in_replay = False; self.current_level = lvl

        # ── GAME_OVER: end of episode, pick next frontier ─────────────────────
        if latest_frame.state == GameState.GAME_OVER:
            self.episode += 1
            f = self._pick_frontier()
            self.replay_seq = f['path'][:] if f else []
            self.replay_idx = 0
            self.in_replay  = bool(self.replay_seq)
            self.curr_path  = []
            if self.episode % 100 == 0:
                max_d   = max((len(v['path']) for v in self.archive.values()), default=0)
                pending = sum(1 for v in self.archive.values() if v['untried'])
                print(f'[GoExplore] ep={self.episode} arch={len(self.archive)} '
                      f'max_depth={max_d} pending={pending}', flush=True)
            return GameAction.RESET

        if latest_frame.state == GameState.NOT_PLAYED:
            return GameAction.RESET

        # ── archive current state ─────────────────────────────────────────────
        h = board_hash(latest_frame)
        avail = [int(a) for a in (getattr(latest_frame, 'available_actions', None) or [1])
                 if 1 <= int(a) <= 7] or [1]

        if h not in self.archive:
            self.archive[h] = {'path': self.curr_path[:], 'untried': set(avail)}
            self.bfs_q.append(h)
        elif len(self.curr_path) < len(self.archive[h]['path']):
            # Shorter path found — update so future replays are more efficient
            self.archive[h]['path'] = self.curr_path[:]

        # ── replay ────────────────────────────────────────────────────────────
        if self.in_replay and self.replay_idx < len(self.replay_seq):
            ai = self.replay_seq[self.replay_idx]
            self.replay_idx += 1; self.curr_path.append(ai); self.total_steps += 1
            if self.replay_idx >= len(self.replay_seq):
                self.in_replay = False
            action = ACTION_MAP.get(ai, GameAction.ACTION1)
            action.reasoning = f'replay {self.replay_idx}/{len(self.replay_seq)}'
            return action

        # ── exploration ───────────────────────────────────────────────────────
        ut = self.archive[h]['untried'] & set(avail)
        ai = random.choice(list(ut)) if ut else random.choice(avail)
        self.archive[h]['untried'].discard(ai)
        self.curr_path.append(ai); self.total_steps += 1
        action = ACTION_MAP.get(ai, GameAction.ACTION1)
        action.reasoning = f'explore depth={len(self.curr_path)} arch={len(self.archive)}'
        return action

    # ── frontier selection ────────────────────────────────────────────────────

    def _pick_frontier(self) -> dict | None:
        """BFS order: shallowest state with untried actions within safe_depth."""
        for h in self.bfs_q:
            v = self.archive.get(h)
            if v and v['untried'] and len(v['path']) < self.safe_depth:
                return v
        # Fallback: ignore depth limit (handles games where safe_depth is too small)
        for h in self.bfs_q:
            v = self.archive.get(h)
            if v and v['untried']:
                return v
        return None  # fully explored

In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import GoExploreAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random":    Random,
    "goexplore": GoExploreAgent,
}
""")

    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent goexplore

In [ ]:
import pandas as pd
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    print('Dummy submission written (non-rerun mode).')